# Create HHMI Awards (agent-browser scrape, no-amount funder)

Creates Howard Hughes Medical Institute investigator records as awards.
HHMI funds **people** (Investigators, Faculty Fellows, etc.) on multi-year
renewable terms rather than discrete projects. There are ~1,739 distinct
investigator profile pages in the HHMI sitemap (current + emeriti combined).

**Prerequisites:**
- Run `scripts/local/hhmi_to_s3.py` to drive agent-browser through every
  /scientists/{slug} URL and upload the parquet.

**Data source:** hhmi.org/scientists/{slug} pages, scraped via
agent-browser (Shadow DOM requires JS rendering — plain HTTP returns
empty shells).
**S3 location:** `s3a://openalex-ingest/awards/hhmi/hhmi_investigators.parquet`

**HHMI funder (OpenAlex):**
- funder_id: 4320306082
- ROR: https://ror.org/006w34k90
- DOI: 10.13039/100000011
- display_name: "Howard Hughes Medical Institute"

**Schema notes — important:**
- HHMI **does not disclose** per-investigator funding amounts publicly.
  `amount` is set to NULL. `currency` is hardcoded "USD" (HHMI is US-
  based) so downstream consumers can identify the missing dimension.
- The Step 6.7 amount/currency check below is **explicitly waived** for
  this funder — pct_amount = 0% is expected.
- One row per (investigator × HHMI program), keyed by the page slug.
- `funder_scheme` = the HHMI role title ("Investigator", "Investigator
  Emeriti", "Hanna H. Gray Fellow", etc.) parsed from the page title.
- `start_year` / `end_year` come from the term in the page title
  (e.g. "2000-2022" → start 2000, end 2022). Active investigators show
  "YYYY-present" → end_year = NULL.
- `lead_investigator.given_name` / `family_name` are **populated** for
  HHMI (unlike most grant funders). The investigator IS the awardee.


## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.hhmi_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/hhmi/hhmi_investigators.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.hhmi_raw;

## Step 1.5: Inspect raw data + verify what's present

In [ ]:
%sql
DESCRIBE openalex.awards.hhmi_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.hhmi_raw LIMIT 5;

In [ ]:
%sql
-- Distribution by role (HHMI program)
SELECT role, COUNT(*) AS cnt
FROM openalex.awards.hhmi_raw
GROUP BY role
ORDER BY cnt DESC;

## Step 2: Transform to award schema

Note: HHMI funds individual scientists. The investigator's name fills
the `lead_investigator` struct directly (unlike most grant funders where
lead_investigator describes the project PI separate from the grantee
organisation). Their host institution is in `affiliation.name`.

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.hhmi_awards
USING delta
AS
WITH hhmi_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320306082
),
src AS (
    SELECT
        url,
        regexp_extract(url, '/scientists/([^/]+)$', 1) AS slug,
        name_full,
        role,
        term,
        start_year,
        end_year,
        institution,
        disciplines,
        bio,
        -- Strip in sequence:
        --   1. trailing degree suffix (PhD/MD/DPhil/DSc/ScD), with or without comma
        --   2. trailing generational suffix (Jr./Sr./II/III/IV), with or without comma
        -- Names without either suffix flow through unchanged.
        --   "William R. Jacobs Jr., PhD" -> "William R. Jacobs"
        --   "David Haussler, PhD"        -> "David Haussler"
        --   "Marta Zlatic"               -> "Marta Zlatic"
        regexp_replace(
            regexp_replace(name_full, ',?\\s*(PhD|MD|DPhil|DSc|ScD)\\.?\\s*$', ''),
            ',?\\s+(Jr|Sr|II|III|IV)\\.?\\s*$', ''
        ) AS name_no_degree
    FROM openalex.awards.hhmi_raw
    WHERE name_full IS NOT NULL
),
parsed_names AS (
    SELECT
        *,
        -- Last whitespace-separated token = family name. Suffixes already stripped above.
        --   "James P. Eisenstein"  -> given="James P."     family="Eisenstein"
        --   "Helmut Schwarz"       -> given="Helmut"        family="Schwarz"
        --   "William R. Jacobs"    -> given="William R."    family="Jacobs"
        CASE
            WHEN array_size(split(trim(name_no_degree), '\\s+')) >= 2
            THEN element_at(split(trim(name_no_degree), '\\s+'), -1)
            ELSE NULL
        END AS family_name_tok,
        CASE
            WHEN array_size(split(trim(name_no_degree), '\\s+')) >= 2
            THEN array_join(slice(split(trim(name_no_degree), '\\s+'), 1, array_size(split(trim(name_no_degree), '\\s+')) - 1), ' ')
            ELSE trim(name_no_degree)
        END AS given_name_tok
    FROM src
)
SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(p.slug)))) % 9000000000 AS id,
    CONCAT(
        COALESCE(p.name_no_degree, p.name_full), ' — HHMI ',
        COALESCE(REGEXP_REPLACE(NULLIF(p.role, 'HHMI'), '^HHMI ', ''), 'Scientist'),
        CASE WHEN p.term IS NOT NULL THEN CONCAT(' (', p.term, ')') ELSE '' END
    ) AS display_name,
    NULLIF(p.bio, '') AS description,
    f.funder_id,
    p.slug AS funder_award_id,
    CAST(NULL AS DOUBLE) AS amount,
    'USD' AS currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    CASE
        WHEN LOWER(p.role) LIKE '%fellow%' THEN 'fellowship'
        ELSE 'research'
    END AS funding_type,
    NULLIF(p.role, 'HHMI') AS funder_scheme,
    'hhmi_scientist_pages' AS provenance,
    CASE WHEN p.start_year IS NOT NULL
         THEN TRY_TO_DATE(CONCAT(p.start_year, '-01-01'), 'yyyy-MM-dd') END AS start_date,
    CASE WHEN p.end_year IS NOT NULL
         THEN TRY_TO_DATE(CONCAT(p.end_year, '-12-31'), 'yyyy-MM-dd') END AS end_date,
    TRY_CAST(p.start_year AS INT) AS start_year,
    TRY_CAST(p.end_year AS INT) AS end_year,
    struct(
        p.given_name_tok AS given_name,
        p.family_name_tok AS family_name,
        CAST(NULL AS STRING) AS orcid,
        CAST(NULL AS DATE) AS role_start,
        struct(
            p.institution AS name,
            CAST(NULL AS STRING) AS country,
            CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
        ) AS affiliation
    ) AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    p.url AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(p.slug)))) % 9000000000) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM parsed_names p
CROSS JOIN hhmi_funder f
WHERE p.slug IS NOT NULL AND TRIM(p.slug) != '';

## Step 3: Insert into openalex_awards_raw at priority 44

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'hhmi_scientist_pages' AND priority = 44;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    44 AS priority
FROM openalex.awards.hhmi_awards;

## Verification

In [ ]:
%sql
SELECT COUNT(*) AS total_hhmi_awards FROM openalex.awards.hhmi_awards;

In [ ]:
%sql
-- Step 6.7 amount/currency check.
-- WAIVED for HHMI: they do not disclose per-investigator amounts.
-- Expect pct_amount = 0%, distinct_currencies = 1 ("USD"), no min/max amount.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies
FROM openalex.awards.hhmi_awards;

In [ ]:
%sql
-- Sample for visual review
SELECT id, SUBSTRING(display_name, 1, 80) AS title,
       funder_scheme AS hhmi_role, start_year, end_year,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name AS institution
FROM openalex.awards.hhmi_awards
ORDER BY start_year DESC LIMIT 15;

In [ ]:
%sql
-- Distribution by HHMI program (role)
SELECT funder_scheme AS hhmi_program, COUNT(*) AS cnt
FROM openalex.awards.hhmi_awards
GROUP BY funder_scheme
ORDER BY cnt DESC;

In [ ]:
%sql
-- Top host institutions
SELECT lead_investigator.affiliation.name AS institution, COUNT(*) AS cnt
FROM openalex.awards.hhmi_awards
WHERE lead_investigator.affiliation.name IS NOT NULL
GROUP BY institution
ORDER BY cnt DESC LIMIT 20;

In [ ]:
%sql
-- Year distribution (start years)
SELECT start_year, COUNT(*) AS cnt
FROM openalex.awards.hhmi_awards
WHERE start_year IS NOT NULL
GROUP BY start_year
ORDER BY start_year DESC;